# Laboratorium 3 — Modelowanie danych w hurtowni

Celem zadania jest przygotowanie danych sprzedażowych Online Retail, zaprojektowanie schematu gwiazdy oraz zaimplementowanie tabel wymiarów i tabeli faktów w Pythonie z użyciem biblioteki Pandas.

Model hurtowni danych będzie zawierał tabelę faktów FactSales oraz wymiary DimCustomer, DimProduct i DimDate.

In [ ]:
import sys
!{sys.executable} -m pip install pandas


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -----------------------------

In [11]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("data/Online_Retail.csv")
OUTPUT_PATH = Path("output")
OUTPUT_PATH.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH, encoding="latin1")

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6.0,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,12/1/10 8:26,3.39,17850.0,United Kingdom


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 140228 entries, 0 to 140227
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    140228 non-null  str    
 1   StockCode    140228 non-null  str    
 2   Description  139770 non-null  str    
 3   Quantity     140227 non-null  float64
 4   InvoiceDate  140227 non-null  str    
 5   UnitPrice    140227 non-null  float64
 6   CustomerID   95724 non-null   float64
 7   Country      140227 non-null  str    
dtypes: float64(3), str(5)
memory usage: 8.6 MB


In [14]:
df.isna().sum()

InvoiceNo          0
StockCode          0
Description      458
Quantity           1
InvoiceDate        1
UnitPrice          1
CustomerID     44504
Country            1
dtype: int64

## Czyszczenie danych

Dane zostały oczyszczone, ponieważ hurtownia danych powinna przechowywać dane nadające się do analiz. Usunięto rekordy bez identyfikatora klienta, anulowane faktury oraz pozycje z niepoprawną ilością lub ceną. Dodano kolumnę Revenue jako miarę sprzedaży.

In [15]:
df_clean = df.copy()

df_clean = df_clean.dropna(subset=["CustomerID"])
df_clean = df_clean[~df_clean["InvoiceNo"].astype(str).str.startswith("C")]
df_clean = df_clean[df_clean["Quantity"] > 0]
df_clean = df_clean[df_clean["UnitPrice"] > 0]

df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
df_clean = df_clean.drop_duplicates()
df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

df_clean.head()

C:\Users\dodom\AppData\Local\Temp\ipykernel_13508\1974504456.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [16]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 92118 entries, 0 to 140226
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    92118 non-null  str           
 1   StockCode    92118 non-null  str           
 2   Description  92118 non-null  str           
 3   Quantity     92118 non-null  float64       
 4   InvoiceDate  92118 non-null  datetime64[us]
 5   UnitPrice    92118 non-null  float64       
 6   CustomerID   92118 non-null  int64         
 7   Country      92118 non-null  str           
 8   Revenue      92118 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(4)
memory usage: 7.0 MB


## Wybór ziarna faktu

Wybrano ziarno na poziomie pojedynczej pozycji faktury. Jeden rekord w tabeli faktów reprezentuje sprzedaż jednego produktu w ramach jednej faktury.

Takie ziarno umożliwia analizę sprzedaży według produktu, klienta i daty. Przykładowo można sprawdzić, które produkty wygenerowały największy przychód w danym miesiącu.

## DimCustomer

Wymiar klienta przechowuje informacje o klientach. Kluczem naturalnym jest CustomerID, natomiast kluczem sztucznym jest CustomerKey.

In [17]:
dim_customer = (
    df_clean[["CustomerID", "Country"]]
    .drop_duplicates()
    .sort_values("CustomerID")
    .reset_index(drop=True)
)

dim_customer.insert(0, "CustomerKey", range(1, len(dim_customer) + 1))

dim_customer.head()

,CustomerKey,CustomerID,Country
0,1,12346,United Kingdom
1,2,12347,Iceland
2,3,12348,Finland
3,4,12350,Norway
4,5,12352,Norway


## DimProduct

Wymiar produktu przechowuje dane opisujące produkty. Kluczem naturalnym jest StockCode, a kluczem sztucznym ProductKey.

In [18]:
dim_product = (
    df_clean[["StockCode", "Description"]]
    .drop_duplicates(subset=["StockCode"])
    .sort_values("StockCode")
    .reset_index(drop=True)
)

dim_product.insert(0, "ProductKey", range(1, len(dim_product) + 1))

dim_product.head()

,ProductKey,StockCode,Description
0,1,10002,INFLATABLE POLITICAL GLOBE
1,2,10080,GROOVY CACTUS INFLATABLE
2,3,10120,DOGGY RUBBER
3,4,10123C,HEARTS WRAPPING TAPE
4,5,10124A,SPOTS ON RED BOOKCOVER TAPE


## DimDate

Wymiar daty pozwala analizować sprzedaż według dnia, miesiąca, kwartału i roku. Kluczem sztucznym jest DateKey.

In [19]:
dim_date = pd.DataFrame({
    "Date": df_clean["InvoiceDate"].dt.date.unique()
})

dim_date["Date"] = pd.to_datetime(dim_date["Date"])
dim_date = dim_date.sort_values("Date").reset_index(drop=True)

dim_date.insert(0, "DateKey", range(1, len(dim_date) + 1))
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["Day"] = dim_date["Date"].dt.day
dim_date["Quarter"] = dim_date["Date"].dt.quarter
dim_date["MonthName"] = dim_date["Date"].dt.month_name()

dim_date.head()

,DateKey,Date,Year,Month,Day,Quarter,MonthName
0,1,2010-12-01,2010,12,1,4,December
1,2,2010-12-02,2010,12,2,4,December
2,3,2010-12-03,2010,12,3,4,December
3,4,2010-12-05,2010,12,5,4,December
4,5,2010-12-06,2010,12,6,4,December


## FactSales

Tabela faktów zawiera miary Quantity, UnitPrice i Revenue. Zamiast kluczy naturalnych wykorzystuje klucze sztuczne z tabel wymiarów: CustomerKey, ProductKey i DateKey.


In [20]:
fact_sales = df_clean.copy()
fact_sales["Date"] = fact_sales["InvoiceDate"].dt.normalize()

fact_sales = fact_sales.merge(
    dim_customer[["CustomerKey", "CustomerID", "Country"]],
    on=["CustomerID", "Country"],
    how="left"
)

fact_sales = fact_sales.merge(
    dim_product[["ProductKey", "StockCode"]],
    on="StockCode",
    how="left"
)

fact_sales = fact_sales.merge(
    dim_date[["DateKey", "Date"]],
    on="Date",
    how="left"
)

fact_sales = fact_sales[
    [
        "InvoiceNo",
        "CustomerKey",
        "ProductKey",
        "DateKey",
        "Quantity",
        "UnitPrice",
        "Revenue"
    ]
]

fact_sales.insert(0, "SalesKey", range(1, len(fact_sales) + 1))

fact_sales.head()

,SalesKey,InvoiceNo,CustomerKey,ProductKey,DateKey,Quantity,UnitPrice,Revenue
0,1,536365,1965,2576,1,6.0,2.55,15.30
1,2,536365,1965,2049,1,6.0,3.39,20.34
2,3,536365,1965,2226,1,8.0,2.75,22.00
3,4,536365,1965,2181,1,6.0,3.39,20.34
4,5,536365,1965,2180,1,6.0,3.39,20.34


In [21]:
fact_sales.isna().sum()

SalesKey       0
InvoiceNo      0
CustomerKey    0
ProductKey     0
DateKey        0
Quantity       0
UnitPrice      0
Revenue        0
dtype: int64

## SCD typu 1 dla DimCustomer

Dla wymiaru klienta zastosowano SCD typu 1. Oznacza to, że w przypadku zmiany kraju klienta poprzednia wartość zostaje nadpisana. Jest to proste rozwiązanie, ale nie zachowuje historii zmian.

In [22]:
def apply_scd_type_1(dim_customer, new_customer_data):
    updated_dim = dim_customer.copy()

    for _, row in new_customer_data.iterrows():
        customer_id = row["CustomerID"]
        new_country = row["Country"]

        mask = updated_dim["CustomerID"] == customer_id

        if mask.any():
            old_country = updated_dim.loc[mask, "Country"].iloc[0]

            if old_country != new_country:
                updated_dim.loc[mask, "Country"] = new_country
        else:
            new_key = updated_dim["CustomerKey"].max() + 1
            new_row = pd.DataFrame([{
                "CustomerKey": new_key,
                "CustomerID": customer_id,
                "Country": new_country
            }])
            updated_dim = pd.concat([updated_dim, new_row], ignore_index=True)

    return updated_dim

In [23]:
test_changes = pd.DataFrame({
    "CustomerID": [17850],
    "Country": ["Poland"]
})

dim_customer_scd1 = apply_scd_type_1(dim_customer, test_changes)

dim_customer_scd1[dim_customer_scd1["CustomerID"] == 17850]

,CustomerKey,CustomerID,Country
1964,1965,17850,Poland


In [24]:
dim_customer.to_csv(OUTPUT_PATH / "DimCustomer.csv", index=False)
dim_product.to_csv(OUTPUT_PATH / "DimProduct.csv", index=False)
dim_date.to_csv(OUTPUT_PATH / "DimDate.csv", index=False)
fact_sales.to_csv(OUTPUT_PATH / "FactSales.csv", index=False)

print("Zapisano tabele do folderu output.")

Zapisano tabele do folderu output.


## Podsumowanie

Zaprojektowano schemat gwiazdy dla danych Online Retail. Centralną tabelą jest FactSales, która zawiera miary sprzedażowe oraz klucze sztuczne do wymiarów DimCustomer, DimProduct i DimDate.

Zastosowanie ziarna na poziomie pozycji faktury daje dużą elastyczność analiz, ponieważ dane można agregować według klienta, produktu oraz czasu. Wadą tego podejścia jest większy rozmiar tabeli faktów.